In [11]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


df = pd.read_csv('eCommerce-dataset.csv')

df.loc[(df['Age'] < 15) | (df['Age'] > 100), 'Age'] = np.nan
df.loc[df['Total_Purchases'] < 0, 'Total_Purchases'] = np.nan
df['Cart_Abandonment_Rate'] = df['Cart_Abandonment_Rate'].clip(upper=100.0)
df['Discount_Usage_Rate'] = df['Discount_Usage_Rate'].clip(upper=100.0)
df['Average_Order_Value'] = df['Average_Order_Value'].clip(
    upper=df['Average_Order_Value'].quantile(0.999)
)

In [12]:
churn_probabilities = model.predict_proba(X)[:, 1]


df_results = df.copy()
df_results['Churn_Probability'] = churn_probabilities


high_risk_customers = df_results[df_results['Churn_Probability'] > 0.70]

high_risk_customers = high_risk_customers.sort_values(
    by='Churn_Probability', ascending=False
)


print(f'Total high-risk customers identified: {len(high_risk_customers)}')
print(
    high_risk_customers[
        ['Customer_Service_Calls', 'Lifetime_Value', 'Churn_Probability']
    ].head(10)
)

Total high-risk customers identified: 10946
       Customer_Service_Calls  Lifetime_Value  Churn_Probability
36056                     6.0         2462.84           0.999501
472                       6.0         2974.48           0.999360
36986                     7.0         4869.08           0.999252
4213                      9.0         2868.88           0.999224
5863                      8.0         2018.36           0.999137
24073                     9.0         2980.43           0.999086
40831                     7.0         4691.98           0.999078
17878                     6.0         2111.86           0.999035
41073                     8.0         2377.19           0.999028
46990                     8.0         2411.02           0.999005


In [13]:
df.loc[(df['Age'] < 15) | (df['Age'] > 100), 'Age'] = np.nan
df.loc[df['Total_Purchases'] < 0, 'Total_Purchases'] = np.nan
df['Cart_Abandonment_Rate'] = df['Cart_Abandonment_Rate'].clip(upper=100.0)
df['Discount_Usage_Rate'] = df['Discount_Usage_Rate'].clip(upper=100.0)
df['Average_Order_Value'] = df['Average_Order_Value'].clip(
    upper=df['Average_Order_Value'].quantile(0.999)
)

In [14]:
X = df.drop(columns=['Churned'])
y = df['Churned']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
num_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        (
            'onehot',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
    ]
)

In [17]:
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'classifier',
            HistGradientBoostingClassifier(
                max_iter=150, learning_rate=0.1, random_state=42
            ),
        ),
    ]
)

In [18]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'Test ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred))

Test ROC-AUC: 0.9280

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      7110
           1       0.92      0.80      0.85      2890

    accuracy                           0.92     10000
   macro avg       0.92      0.88      0.90     10000
weighted avg       0.92      0.92      0.92     10000

